# K4 — route-based interface pick

**Checks:** with no manual interface, the sniffer binds the adapter the OS routes through, and says so in the
system tab: `Auto-Targeting Network Interface: <ip> (default route | first physical adapter)`.

1. **Two live adapters** (Ethernet + Wi-Fi) → `(default route)`, the adapter the game uses, capture works.
2. **Full-tunnel VPN on** → the *physical* adapter, `(first physical adapter)`, capture still works.
   **This is the open question:** if chat is only seen on the VPN adapter, the "virtual adapters are not trusted" rule is wrong.
3. **Offline** → no crash, falls back to the list.

The notebook reads `route print` and PowerShell's adapter list itself; you start the app and **paste the system-tab line**.
Run Jupyter as Administrator. Make sure *no manual interface* is set in the app's settings.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import common
from runbook.common import Recorder, capture, ask_text
from runbook import interface
rec = Recorder("interface")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")

KEYWORDS = interface.virtual_keywords((common.ROOT.parent / "crates/core/src/sniffer_net.rs").read_text(encoding="utf-8"))
print("the app's virtual-adapter keywords:", KEYWORDS)

def snapshot(route_fixture, adapters_fixture):
    code, route_text = capture(["route", "print", "-4"], fixture=route_fixture)
    routes = interface.parse_default_routes(route_text)
    code2, adapter_text = capture(["powershell", "-NoProfile", "-Command", interface.POWERSHELL_ADAPTERS],
                                  fixture=adapters_fixture)
    adapters = interface.parse_adapters(adapter_text)
    print("default routes:", [(r.interface_ip, r.metric) for r in routes])
    for a in adapters:
        print(f"  {a.ip:15} {a.alias}  [{a.description}]  status={a.status} virtual={a.virtual}")
    return routes, adapters

def record_all(check, checks):
    for title, status, evidence in checks:
        rec.record(check, title, status, evidence)

## 1 · Two live adapters
Connect **both** Ethernet and Wi-Fi. Start (or restart) the app so the sniffer binds again; open the system tab.

In [ ]:
line1 = ask_text("LOG1", "Paste the system-tab line 'Auto-Targeting Network Interface: ...': ")
routes, adapters = snapshot("route_print_two_adapters.txt", "adapters_two.json")
record_all("K4-1", interface.check_default_route(interface.parse_log_line(line1), routes, adapters, KEYWORDS))

In [ ]:
rec.manual("K4-1", "two adapters: chat is captured without the manual picker",
           "Log into the game with both adapters up. Leave the interface setting on automatic.",
           "chat lines appear in the app")

## 2 · Full-tunnel VPN
Turn on a VPN that routes **all** traffic (not split tunnel). **Restart the sniffer** (or the app): the adapter is picked at start.

In [ ]:
line2 = ask_text("LOG2", "Paste the system-tab line again, with the VPN on: ")
routes, adapters = snapshot("route_print_vpn.txt", "adapters_vpn.json")
record_all("K4-2", interface.check_vpn(interface.parse_log_line(line2), routes, adapters, KEYWORDS))

In [ ]:
rec.manual("K4-2", "VPN on: chat is still captured (THE OPEN QUESTION)",
           "Log into the game with the VPN on and watch the chat tab for a minute.",
           "chat lines appear. If they do NOT, answer `fail: packets only on the VPN adapter` -- the virtual-adapter rule is wrong")

## 3 · Offline
Turn the VPN off and disconnect the network completely (Wi-Fi off, cable out). Start the app.

In [ ]:
rec.manual("K4-3", "offline: no crash, falls back to the list",
           "Start the app with no network at all. Open the system tab: read what the sniffer says. "
           "Then reconnect and restart the sniffer.",
           "the app stays up, the system tab says which adapter or reports no interface; after reconnecting it binds again")

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())